# 消息

上一章我们封装了大模型调用器，可以应对不同的大模型。但是每次只能传入一句提问，无法满足小明对学习助手的要求：

- 它应该知道自己的责任：是这本书的学习助手，面对的是编程新手；
- 它应该有适合的风格：比如使用中文、讲得简单一点；
- 它最好还能看到之前聊过的内容，可以更针对性地输出答案。

这些都不是“一句话”能装下的。真实的对话，由多轮、多条**消息**组成，每条消息属于不同的**角色**，标识这句话是谁说的。这一章，我们就来认识对话里最基本的三种角色。

## 消息角色

| 角色 | 是谁说的 | 例子 |
|---|---|---|
| system（系统） | 开发者，也就是我们 | “你是一个学习助手，回答要简短，适合新手。” |
| user（用户） | 使用者，比如小明 | “什么是变量？” |
| assistant（助手） | 大模型 | “变量就像一个贴了名字的盒子……” |

一段对话发给大模型时，大致会是这个样子：

```
system:    你是一个学习助手，回答要简短，适合新手。
user:      什么是变量？
assistant: 变量就像一个贴了名字的盒子，可以用来存放数据。
user:      那常量呢？
```

user 和 assistant 很好理解，就是一问一答。最特别的是 **system**。

## 系统消息

系统消息是开发者写给大模型的“身份说明”，用来设定它**是谁、做什么、怎么做**。使用者一般看不到它，但它对整段对话都有效。

系统消息里通常可以写这些内容：

- **身份**：你是谁。比如“你是《从大模型到智能体》这本书的学习助手”；
- **任务**：你要做什么。比如“帮助初学者学习如何构建 AI 智能体”；
- **风格**：怎么回答。比如“用中文回答，尽量简短，多举例子”；
- **规则**：不能做什么。比如“不要直接给出练习题的答案，要引导学生自己思考”。

写系统消息有几个小技巧：

- **具体**：“回答简短”不如“每次回答不超过三句话”；
- **简洁**：把最重要的要求写清楚就行，写得太长太杂，模型反而抓不住重点；
- **一次只改一处**：效果不理想时，改一个地方，看看效果，再改下一个地方。

同样的模型，换一条系统消息，就能变成一个风格完全不同的助手。

In [1]:
import os
from abc import ABC, abstractmethod
from enum import Enum
from typing import List

import anthropic
from anthropic.types import UserMessageParam, AssistantMessageParam
from openai import OpenAI
from openai.types.chat import ChatCompletionSystemMessageParam, ChatCompletionUserMessageParam, ChatCompletionAssistantMessageParam

## 消息类

### 角色

首先用一个**枚举**（`Enum`）列出目前用到的三种角色：

In [2]:
class MsgType(Enum):
    SYSTEM = "System"
    USER = "User"
    ASSISTANT = "Assistant"

### 抽象类

然后是所有消息的基础抽象类 `Message`。它规定了每条消息都要有的两个信息：

- `msg_type`：这条消息的角色。
- `summary()`：返回这条消息的文字内容。不管是哪种消息，都可以用 `summary()` 拿到它的文字。

In [3]:
class Message(ABC):

    def __init__(self, msg_type: MsgType):
        self.msg_type = msg_type

    @abstractmethod
    def summary(self) -> str:
        pass

    def __repr__(self) -> str:
        return f"Message({self.msg_type.value}): {self.summary()}"

三种角色各有一个子类。它们的结构都一样：在 `__init__` 里设置自己的角色，再保存消息的文字。为了让代码读起来更清楚，文字字段的名字各不相同：系统消息叫 `description`（描述），用户消息叫 `request`（请求），助手消息叫 `content`（内容）。

### 系统消息类

系统对大模型角色的定义。

In [4]:
class SystemMessage(Message):

    def __init__(self, description: str):
        super().__init__(MsgType.SYSTEM)
        self.description = description

    def summary(self) -> str:
        return self.description

### 用户消息类

用户的提问。

In [5]:
class UserMessage(Message):

    def __init__(self, request: str):
        super().__init__(MsgType.USER)
        self.request = request

    def summary(self) -> str:
        return self.request

### 助手消息类

大模型的回答。

In [6]:
class AssistantMessage(Message):

    def __init__(self, content: str):
        super().__init__(MsgType.ASSISTANT)
        self.content = content

    def summary(self) -> str:
        return self.content

现在，让我们给小明的学习助手写一个系统消息：

In [7]:
system_message = SystemMessage("You're a learning assistant for the book 《从大模型到智能体》 (From LLM to Agent). "
                               "You help beginners learn how to build AI agents.")
user_message = UserMessage("你好，我叫小明，正在自学编程。请用两三句话介绍一下你自己。")

## 大模型调用器

### 基础类

`LLMProvider` 的 `call` 也要跟着升级：

- 传入一条系统消息 `system` 和一个消息列表 `messages`；
- 返回一条助手消息。

In [8]:
class LLMProvider(ABC):

    @abstractmethod
    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        pass

### OpenAI 调用器

`OpenAIProvider` 多了两个“翻译”方法：

- `_message_schema`：把我们的消息列表，翻译成 OpenAI 要求的格式。用 `isinstance` 判断每条消息是哪个类，再转换成对应的 `role`。
- `_response_schema`：把 OpenAI 返回的结果，翻译回我们的 `AssistantMessage`。

注意 `call` 的第一行：如果有系统消息，就把它放到消息列表的最前面。这是 OpenAI 处理系统消息的方式。`max_tokens` 对应 OpenAI 的参数 `max_completion_tokens`，表示最多生成多少 token（最基础的推理停止条件）。

In [9]:
class OpenAIProvider(LLMProvider):

    def __init__(self):
        self.client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=None
        )
        self.default_model = "gpt-5.4-mini"

    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        if system:
            messages = [system] + messages
        kwargs = dict(
            model=model or self.default_model,
            max_completion_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        response = self.client.chat.completions.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, SystemMessage):
                schema.append(ChatCompletionSystemMessageParam(role="system", content=msg.description))
            elif isinstance(msg, UserMessage):
                schema.append(ChatCompletionUserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                schema.append(ChatCompletionAssistantMessageParam(role="assistant", content=msg.content))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        message = response.choices[0].message
        return AssistantMessage(message.content or "")

In [10]:
provider = OpenAIProvider()
message = provider.call(system_message, [user_message])
print(message.summary())

你好，小明！我是你的学习助手，专门帮助你理解大模型和智能体相关的知识，也可以陪你一起从零学编程、做练习和解决问题。  
如果你愿意，我可以用很简单的方式，带你一步一步学习《从大模型到智能体》里的内容。


### Claude 调用器

`ClaudeProvider` 也有同样的两个翻译方法，但处理系统消息的方式不一样：系统消息不放进 `messages`，而是放到单独的 `system` 参数里。所以它的 `_message_schema` 只需要处理用户消息和助手消息。

另外，Claude 返回的 `response.content` 是一个列表，里面可能有好几段内容。我们把其中所有文字（`type == "text"`）拼起来，作为回答。

In [11]:
class ClaudeProvider(LLMProvider):

    def __init__(self):
        self.client = anthropic.Anthropic(
            api_key=os.getenv("ANTHROPIC_API_KEY")
        )
        self.default_model = "claude-haiku-4-5"

    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        kwargs = dict(
            model=model or self.default_model,
            max_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        if system:
            kwargs["system"] = system.summary()
        response = self.client.messages.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, UserMessage):
                schema.append(UserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                schema.append(AssistantMessageParam(role="assistant", content=msg.content))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        text = "".join(block.text for block in response.content if block.type == "text")
        return AssistantMessage(text)

In [12]:
provider = ClaudeProvider()
message = provider.call(system_message, [user_message])
print(message.summary())

你好，小明！很高兴认识你。👋

我是这本《从大模型到智能体》的学习助手，专门帮助像你这样的编程初学者理解和掌握大模型、提示工程、智能体等核心概念，以及如何动手构建自己的AI应用。无论你对哪些章节有疑问，或需要代码示例和学习建议，都欢迎随时问我！


## 大模型调用

最后做个小实验：同一个问题，配上不同的系统消息，看看回答有什么不同。

In [13]:
question = UserMessage("什么是变量？")
for style in ["用一句话回答，适合完全没学过编程的人。",
              "像一位耐心的老师那样回答，并举一个 Python 的小例子。"]:
    reply = provider.call(SystemMessage(style), [question])
    print(f"--- system: {style}\n{reply.summary()}\n")

--- system: 用一句话回答，适合完全没学过编程的人。
# 变量就是一个"装东西的盒子"

想象你有一个标签盒子，你可以往里面放数字、文字或其他信息，需要时就打开盒子取出来用。

**举个例子：**
```
年龄 = 18
名字 = "小明"
```

这里"年龄"和"名字"就是两个盒子，一个装着数字18，一个装着文字"小明"。程序可以随时记住、改变或使用这些信息。

--- system: 像一位耐心的老师那样回答，并举一个 Python 的小例子。
# 什么是变量？

## 简单的理解

**变量就像一个装东西的盒子**。你可以在盒子里放入数据（数字、文字等），给这个盒子起个名字，以后需要时就可以打开盒子取出来用。

## 具体来说

变量有三个基本要素：
1. **名字** - 盒子的标签
2. **数据** - 盒子里装的东西
3. **地址** - 盒子在内存中的位置

## Python 例子

```python
# 创建变量并赋值
age = 25
name = "小王"
score = 95.5

# 使用变量
print(name)  # 输出：小王
print(age)   # 输出：25
print(score) # 输出：95.5

# 修改变量的值
age = 26  # 年纪增长了
print(age)  # 输出：26
```

## 再举个生活比喻

想象你有一个笔记本：
- 📓 笔记本就是**变量**
- 📝 笔记本上写的内容就是**数据**
- 📌 笔记本的标签（比如"购物清单"）就是**变量名**

```python
购物清单 = "牛奶、面包、鸡蛋"
print(购物清单)  # 拿出笔记本看看里面写了什么
```

是不是很简单呢？ 😊 有什么不懂的可以继续问我！



模型、问题都没变，只换了系统消息，回答的长短和风格就完全不一样了。

## 动手试试

- 修改学习助手的系统消息，比如加上“每次回答结尾，提一个问题帮学生检查自己是否理解”，看看效果。

## 小结

- 对话由多条**消息**组成，每条消息有一个**角色**：system（开发者的设定）、user（使用者）、assistant（大模型）；
- **系统消息**设定助手的身份、任务、风格和规则，对整段对话都有效。写得具体、简洁，效果最好；
- 我们把消息包装成 `SystemMessage`、`UserMessage`、`AssistantMessage` 三个类：程序只和这些类打交道，各家的格式由 Provider 负责翻译，以后加新功能也更方便；
- OpenAI 把系统消息放在 `messages` 里，Claude 用单独的 `system` 参数，这些差别都隐藏在了 Provider 里面。

到这里，学习助手已经有了身份和风格，可每次还是只能回答一个问题。下一章，我们将真正开始构建**智能体**，让小明可以和它一直聊下去。